# Cafe revenue: starter notebook

**Task.** Predict the daily revenue (million VND) of each store for every day in `test.csv`, from 1 January to 31 March 2026.

**Metric.** RMSLE, the root mean squared error of `log(1 + revenue)`. It measures *relative* errors: being 10 % off costs about the same for a small store and for a big one.

This notebook builds a simple baseline (the median revenue of each store on each weekday). Several tables are provided and the data is messy on purpose; improving on this baseline is your job.

**Setup.** Import the libraries and find the data folder (the same code works on Kaggle and on your laptop).

In [1]:
import numpy as np
import pandas as pd

import os, glob
# works on Kaggle (/kaggle/input/...) and locally (files next to the notebook)
hits = (glob.glob("/kaggle/input/**/stores.csv", recursive=True)   # on Kaggle
        + glob.glob("stores.csv")                                  # next to the notebook
        + glob.glob("data/stores.csv"))                            # in a data/ folder
DATA = (os.path.dirname(hits[0]) or ".") if hits else "."
print("data folder:", DATA)

data folder: .


**Load the five tables.** `parse_dates` turns the text dates into real dates. The loop prints the size and the date range of each table.

In [2]:
train = pd.read_csv(f"{DATA}/train.csv", parse_dates=["date"])        # daily revenue
test = pd.read_csv(f"{DATA}/test.csv", parse_dates=["date"])          # days to predict
stores = pd.read_csv(f"{DATA}/stores.csv")                            # one row per store
weather = pd.read_csv(f"{DATA}/weather.csv", parse_dates=["date"])    # per city and day
holidays = pd.read_csv(f"{DATA}/holidays.csv", parse_dates=["date"])  # special days
tables = [("train", train), ("test", test), ("stores", stores),
          ("weather", weather), ("holidays", holidays)]
for name, df in tables:
    if "date" in df:
        print(f"{name:9s}", df.shape, df.date.min().date(), "to", df.date.max().date())
    else:
        print(f"{name:9s}", df.shape)

train     (26632, 4) 2024-01-01 to 2025-12-31
test      (3869, 4) 2026-01-01 to 2026-03-31
stores    (45, 8)
weather   (1593, 4) 2024-01-01 to 2026-03-31
holidays  (55, 3) 2024-01-01 to 2026-03-08


**The target.** Summary of the revenue column. Does every value look plausible?

In [3]:
train.revenue.describe().round(1)

count    26632.0
mean        51.3
std       1015.9
min          0.6
25%         12.4
50%         18.3
75%         24.1
max      53600.0
Name: revenue, dtype: float64

**The stores table.** Store type, size, opening date… Look at how each column is written.

In [4]:
stores.head(8)

,store_id,city,district,store_type,area_m2,seats,has_parking,open_date
0,S01,Ha Noi,Ba Dinh,university,149.0,62,1,2025-12-07
1,S02,Ha Noi,Thanh Xuan,residential,95.0,28,1,27/07/2022
2,S03,Ha Noi,Hai Ba Trung,office,120.0,38,0,12/03/2018
3,S04,Ha Noi,Tay Ho,street,101.0,41,0,17/04/2019
4,S05,Ha Noi,Hai Ba Trung,street,191.0,75,1,2020-12-24
5,S06,Ha Noi,Dong Da,office,111.0,53,1,2019-11-16
6,S07,Ha Noi,Thanh Xuan,office,111.0,36,1,2025-06-10
7,S08,Ha Noi,Nam Tu Liem,street,138.0,54,0,2025-05-09


## A time-based validation

The test period is the *future*, so we hold out the last three months of 2025 and train on the rest. A random split would let the model peek at days that come after the ones it predicts.

In [5]:
def rmsle(y, p):
    # clip: a prediction below 0 is not allowed, log1p(x) = log(1 + x)
    return np.sqrt(np.mean((np.log1p(np.clip(p, 0, None)) - np.log1p(y)) ** 2))

fit = train[train.date < "2025-10-01"]       # history the model may use
val = train[train.date >= "2025-10-01"]      # Oct-Dec 2025, plays the role of the test

## Baseline: median revenue per store and weekday
For each (store, weekday) pair, take the median revenue in the history. If a pair is missing, fall back to the store's median, then to the overall median.

In [6]:
def weekday_median(history, target):
    h = history.assign(dow=history.date.dt.dayofweek)     # 0 = Monday ... 6 = Sunday
    m = h.groupby(["store_id", "dow"]).revenue.median()   # one value per (store, weekday)
    keys = list(zip(target.store_id, target.date.dt.dayofweek))
    p = pd.Series(keys).map(m)                            # look up each target day
    store_med = target.store_id.map(h.groupby("store_id").revenue.median())
    p = p.fillna(store_med.reset_index(drop=True))        # fallback 1: store median
    return p.fillna(h.revenue.median()).values            # fallback 2: overall median

print("validation RMSLE:", round(rmsle(val.revenue.values, weekday_median(fit, val)), 4))

validation RMSLE: 0.2387


**Submission.** Recompute the medians on all of `train` and write `submission.csv` with the two columns Kaggle expects: `id`, `revenue`.

In [7]:
sub = pd.DataFrame({"id": test.id, "revenue": weekday_median(train, test)})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,revenue
0,100000,11.880
1,100001,18.175
2,100002,17.120
3,100003,28.770
4,100004,18.510


## Where to go next

- Check the revenue column store by store. Do some values look wrong?
- Join the stores, weather and holidays tables, and clean them first.
- Think about what is special in the test period, and whether your validation period contains it.
- Some stores and some days behave differently from the rest; find out which, and why.
- Model the log of revenue, and add calendar, store and weather features.